# UPI Fraud Ring & Merchant Analytics

## 01 - Data Understanding

### Objective
Understand the structure, quality, completeness, and relationships
among the UPI transactions, KYC records, merchant master, and
chargeback datasets before performing data cleaning.

### Datasets
1. UPI Transactions
2. KYC Records
3. Merchant Master
4. Chargebacks


### Tools
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully!")

Libraries imported successfully!


In [14]:
import os

print(os.getcwd())

C:\Users\VIKASH\Desktop\UPI-Fraud-Ring-Merchant-Analytics\notebooks


In [15]:
transactions = pd.read_csv(
    "../data/raw/track1_upi_transactions.csv"
)

kyc = pd.read_csv(
    "../data/raw/track1_kyc_records.csv"
)

merchants = pd.read_csv(
    "../data/raw/track1_merchants_master.csv"
)

chargebacks = pd.read_json(
    "../data/raw/track1_chargebacks.json"
)

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [16]:
print("Transactions:", transactions.shape)
print("KYC:", kyc.shape)
print("Merchants:", merchants.shape)
print("Chargebacks:", chargebacks.shape)

Transactions: (20400, 8)
KYC: (36400, 12)
Merchants: (6210, 11)
Chargebacks: (2884, 13)


## 3. Dataset Structure

Before cleaning the data, we inspect the columns, data types,
and sample records from each dataset.

In [17]:
print("========== TRANSACTIONS ==========")
print(transactions.columns.tolist())

print("\n========== KYC ==========")
print(kyc.columns.tolist())

print("\n========== MERCHANTS ==========")
print(merchants.columns.tolist())

print("\n========== CHARGEBACKS ==========")
print(chargebacks.columns.tolist())

========== TRANSACTIONS ==========
['txn_id', 'timestamp', 'user_id', 'merchant_id', 'amount', 'utr', 'mcc', 'status']

========== KYC ==========
['user_id', 'full_name', 'pan', 'aadhaar', 'date_of_birth', 'city', 'state', 'monthly_income', 'occupation', 'signup_timestamp', 'kyc_status', 'risk_segment']

========== MERCHANTS ==========
['merchant_id', 'merchant_name', 'mcc', 'merchant_category', 'business_type', 'city', 'state', 'onboarding_date', 'settlement_account', 'merchant_status', 'declared_avg_ticket_size']

========== CHARGEBACKS ==========
['complaint_id', 'txn_id', 'user_id', 'merchant_id', 'transaction_timestamp', 'reported_timestamp', 'disputed_amount', 'reason_code', 'complaint_text', 'resolution_status', 'bank_response_timestamp', 'severity', 'channel']


In [18]:
print("========== TRANSACTIONS ==========")
display(transactions.head())

print("========== KYC ==========")
display(kyc.head())

print("========== MERCHANTS ==========")
display(merchants.head())

print("========== CHARGEBACKS ==========")
display(chargebacks.head())

========== TRANSACTIONS ==========


,txn_id,timestamp,user_id,merchant_id,amount,utr,mcc,status
0,TXN00011869,2026-01-15 00:11:30,USR45826,MCH7045,15722.34,UTR6498104698,5411.0,COMPLETED
1,TXN00010383,2026-01-17 20:09:44,USR79397,MCH5031,Rs. 6362.9,UTR7656190355,4131.0,TXN_FAILED
2,TXN00008297,2026-01-23 14:10:16,USR87810,MCH9809,15446.19,UTR9037001889,5411.0,S
3,TXN00006448,1770063471,USR54287,MCH6928,12110.49,UTR5257823698,5411.0,S
4,TXN00018792,2026-03-31 14:02:37,USR53865,MCH8121,Rs. 19432.94,UTR4204272894,4131.0,TXN_SUCCESS


========== KYC ==========


,user_id,full_name,pan,aadhaar,date_of_birth,city,state,monthly_income,occupation,signup_timestamp,kyc_status,risk_segment
0,USR16112,Dhriti Deshmukh,SEJAA8194O,715658320763,06/04/1967 12:14 AM,Bombay,Maharashtra,35119,Retired,2025-12-02 02:18:16,Done,LOW
1,USR17216,Megha Jani,QT0ZZ5561X,023412025028,NaN,Lucknow,Uttar Pradesh,80907,Freelancer,01-31-2024,Verified,medium
2,USR 45454,PANINI LAL,QCNTL9489O,2781 6299 9816,NaN,kolkata,West Bengal,"₹11,214",Farmer,2026-01-30 23:57:18,Pending,High
3,USR46189,Jack Parikh,RRVUI4059Q,0667 8032 7731,NaN,Amritsar,Punjab,27.3k,Student,09-04-2025,VERIFIED,low
4,USR85256,Eta Ravi,ygvoi9236w,176258817110,1969-02-19 08:25:23,delhi,Delhi,"INR 24,705",Salaried,03-Feb-2026,APPROVED,LOW


========== MERCHANTS ==========


,merchant_id,merchant_name,mcc,merchant_category,business_type,city,state,onboarding_date,settlement_account,merchant_status,declared_avg_ticket_size
0,mch2849,"BHAVSAR, KOTA AND ZACHARIA",MCC-7011,hotel_lodging,Private Limited,Ludhiana,Punjab,09-23-2025,NaN,Inactive,"INR 2,432.18"
1,MCH4314,Deshmukh Ltd,5699,Apparel,PRIVATE_LIMITED,Jalandhar,Punjab,NaN,3021439858,I,986.05
2,MCH1986,"Bains, Chanda and Gh0sh",5311,Retail,individual,Jalandhar,Punjab,10/01/2026,NaN,A,"INR 1,427.52"
3,mch3899,Goswami-Bath,4131,Transportation,Sole Proprietor,Hyderabad,Telangana,14-Jan-2023,NaN,ACTIVE,-1271.48
4,MCH4859,VASA-RAJU,5812,Restaurant,SOLE_PROPRIETOR,Hyd,Telangana,1742180385,XXXX9523,A,Rs. 214


========== CHARGEBACKS ==========


,complaint_id,txn_id,user_id,merchant_id,transaction_timestamp,reported_timestamp,disputed_amount,reason_code,complaint_text,resolution_status,bank_response_timestamp,severity,channel
0,CBK0002082,TXN00004325,usr97580,mch1127,2026/01/28,02-01-2026,,Merchant Not Delivered,Customer says amount was debited twice.,CLOSED,2026-02-10 03:19:10,Critical,ivr
1,CBK0001941,TXN00003720,USR54113,3835,25/02/2026 10:24 AM,1772691855,414.69,login compromised,User reports money deducted but merchant denie...,In Progress,12/03/2026 06:24 AM,H,Call Center
2,CBK0001799,TXN00012539,USR17980,mch3700,2026/02/04,02-05-2026,"Rs. 7,039",customer issue,merchant service was not delivered after payment.,OPEN,06/03/2026,P4,IVR
3,CBK0002465,TXN00017802,USR76148,MCH4534,30-Mar-2026,01-Apr-2026,1303.05,no service,Suspicious high-value payment disputed by cust...,Rejected,07-Apr-2026,H,ivr
4,CBK0001870,TXN00015944,USR24660,MCH1686,09-Jan-2026,1768424501,1459.42,merchant service issue,Merchant service was not delivered after payment.,In Progress,01-26-2026,Low,App


## 4. Data Types

We inspect the data types of all columns to identify fields that
require conversion during the cleaning stage.

In [19]:
print("========== TRANSACTIONS DATA TYPES ==========")
display(transactions.dtypes)

print("\n========== KYC DATA TYPES ==========")
display(kyc.dtypes)

print("\n========== MERCHANTS DATA TYPES ==========")
display(merchants.dtypes)

print("\n========== CHARGEBACKS DATA TYPES ==========")
display(chargebacks.dtypes)

========== TRANSACTIONS DATA TYPES ==========


txn_id             str
timestamp          str
user_id            str
merchant_id        str
amount             str
utr                str
mcc            float64
status             str
dtype: object


========== KYC DATA TYPES ==========


user_id             str
full_name           str
pan                 str
aadhaar             str
date_of_birth       str
city                str
state               str
monthly_income      str
occupation          str
signup_timestamp    str
kyc_status          str
risk_segment        str
dtype: object


========== MERCHANTS DATA TYPES ==========


merchant_id                 str
merchant_name               str
mcc                         str
merchant_category           str
business_type               str
city                        str
state                       str
onboarding_date             str
settlement_account          str
merchant_status             str
declared_avg_ticket_size    str
dtype: object


========== CHARGEBACKS DATA TYPES ==========


complaint_id                  str
txn_id                        str
user_id                       str
merchant_id                   str
transaction_timestamp         str
reported_timestamp            str
disputed_amount            object
reason_code                   str
complaint_text                str
resolution_status             str
bank_response_timestamp       str
severity                      str
channel                       str
dtype: object

## 5. Missing Value Analysis

Missing values can affect fraud detection, customer profiling,
merchant analysis, and transaction-level joins. We therefore
measure missing values in every dataset before cleaning.

In [20]:
print("========== TRANSACTIONS MISSING VALUES ==========")
display(transactions.isnull().sum())

print("\n========== KYC MISSING VALUES ==========")
display(kyc.isnull().sum())

print("\n========== MERCHANTS MISSING VALUES ==========")
display(merchants.isnull().sum())

print("\n========== CHARGEBACKS MISSING VALUES ==========")
display(chargebacks.isnull().sum())

========== TRANSACTIONS MISSING VALUES ==========


txn_id            0
timestamp         0
user_id           0
merchant_id       0
amount            0
utr            1024
mcc            2926
status            0
dtype: int64


========== KYC MISSING VALUES ==========


user_id                0
full_name              0
pan                 1896
aadhaar             2664
date_of_birth       2944
city                   0
state                  0
monthly_income      2933
occupation             0
signup_timestamp    2910
kyc_status             0
risk_segment           0
dtype: int64


========== MERCHANTS MISSING VALUES ==========


merchant_id                    0
merchant_name                  0
mcc                          514
merchant_category              0
business_type                  0
city                           0
state                          0
onboarding_date              499
settlement_account          2451
merchant_status                0
declared_avg_ticket_size     371
dtype: int64


========== CHARGEBACKS MISSING VALUES ==========


complaint_id               0
txn_id                     0
user_id                    0
merchant_id                0
transaction_timestamp      0
reported_timestamp         0
disputed_amount            0
reason_code                0
complaint_text             0
resolution_status          0
bank_response_timestamp    0
severity                   0
channel                    0
dtype: int64

In [21]:
def missing_summary(df):
    summary = pd.DataFrame({
        "Missing_Count": df.isnull().sum(),
        "Missing_Percentage": (df.isnull().mean() * 100).round(2)
    })
    return summary.sort_values("Missing_Count", ascending=False)


print("========== TRANSACTIONS ==========")
display(missing_summary(transactions))

print("========== KYC ==========")
display(missing_summary(kyc))

print("========== MERCHANTS ==========")
display(missing_summary(merchants))

print("========== CHARGEBACKS ==========")
display(missing_summary(chargebacks))

========== TRANSACTIONS ==========


,Missing_Count,Missing_Percentage
mcc,2926,14.34
utr,1024,5.02
txn_id,0,0.00
timestamp,0,0.00
merchant_id,0,0.00
user_id,0,0.00
amount,0,0.00
status,0,0.00


========== KYC ==========


,Missing_Count,Missing_Percentage
date_of_birth,2944,8.09
monthly_income,2933,8.06
signup_timestamp,2910,7.99
aadhaar,2664,7.32
pan,1896,5.21
full_name,0,0.00
user_id,0,0.00
city,0,0.00
state,0,0.00
occupation,0,0.00


========== MERCHANTS ==========


,Missing_Count,Missing_Percentage
settlement_account,2451,39.47
mcc,514,8.28
onboarding_date,499,8.04
declared_avg_ticket_size,371,5.97
merchant_id,0,0.00
merchant_name,0,0.00
merchant_category,0,0.00
state,0,0.00
city,0,0.00
business_type,0,0.00


========== CHARGEBACKS ==========


,Missing_Count,Missing_Percentage
complaint_id,0,0.0
txn_id,0,0.0
user_id,0,0.0
merchant_id,0,0.0
transaction_timestamp,0,0.0
reported_timestamp,0,0.0
disputed_amount,0,0.0
reason_code,0,0.0
complaint_text,0,0.0
resolution_status,0,0.0


## 6. Duplicate Record Analysis

Duplicate records can inflate transaction volume, customer counts,
merchant activity, and chargeback metrics. We identify exact duplicate
rows before deciding how they should be handled during cleaning.

In [22]:
print("Duplicate transaction rows:", transactions.duplicated().sum())
print("Duplicate KYC rows:", kyc.duplicated().sum())
print("Duplicate merchant rows:", merchants.duplicated().sum())
print("Duplicate chargeback rows:", chargebacks.duplicated().sum())

Duplicate transaction rows: 400
Duplicate KYC rows: 278
Duplicate merchant rows: 12
Duplicate chargeback rows: 84


In [23]:
print("========== TRANSACTIONS ==========")
print("Unique txn_id:", transactions["txn_id"].nunique())
print("Unique user_id:", transactions["user_id"].nunique())
print("Unique merchant_id:", transactions["merchant_id"].nunique())

print("\n========== KYC ==========")
print("Unique user_id:", kyc["user_id"].nunique())

print("\n========== MERCHANTS ==========")
print("Unique merchant_id:", merchants["merchant_id"].nunique())

print("\n========== CHARGEBACKS ==========")
print("Unique complaint_id:", chargebacks["complaint_id"].nunique())
print("Unique txn_id:", chargebacks["txn_id"].nunique())

========== TRANSACTIONS ==========
Unique txn_id: 20000
Unique user_id: 17878
Unique merchant_id: 8051

========== KYC ==========
Unique user_id: 32165

========== MERCHANTS ==========
Unique merchant_id: 5083

========== CHARGEBACKS ==========
Unique complaint_id: 2800
Unique txn_id: 2582


## 7. Initial Data Quality Findings

The initial exploration identified several data-quality issues that
must be addressed before fraud and merchant analytics.

### Key Findings

1. **Duplicate records**
   - Transactions contain 400 exact duplicate rows.
   - KYC records contain 278 exact duplicate rows.
   - Merchant records contain 12 exact duplicate rows.
   - Chargeback records contain 84 exact duplicate rows.

2. **Missing values**
   - Transactions contain 2,926 missing MCC values and 1,024 missing UTR values.
   - KYC records contain missing PAN, Aadhaar, date of birth,
     monthly income, and signup timestamp values.
   - Merchant records contain missing settlement accounts, MCC,
     onboarding dates, and declared average ticket sizes.
   - Chargeback records contain no missing values in the inspected fields.

3. **Inconsistent formats**
   - Transaction timestamps appear in mixed date/time and epoch formats.
   - Transaction amounts contain different representations, including
     currency-formatted values.
   - User and merchant IDs have inconsistent capitalization.
   - KYC dates contain multiple formats and missing values.
   - Merchant MCC values appear in different formats such as `MCC-7011`
     and numeric values.
   - Chargeback timestamps also contain mixed date and epoch formats.

4. **Non-unique entity identifiers**
   - The transaction dataset contains 20,000 unique transaction IDs
     across 20,400 rows.
   - Transactions contain 17,878 unique users and 8,051 unique merchants.
   - The KYC dataset contains 32,165 unique user IDs across 36,400 rows.
   - The merchant dataset contains 5,083 unique merchant IDs across 6,210 rows.
   - The chargeback dataset contains 2,800 unique complaint IDs and
     2,582 unique transaction IDs.

### Conclusion

The raw datasets require systematic data cleaning before analytical
modelling. The cleaning process will normalize identifiers, standardize
amounts and timestamps, validate transaction and reference fields,
handle duplicates, preserve useful records with missing values, and
validate relationships between transactions, KYC, merchants, and
chargebacks.

## 8. Data Understanding Summary

The four datasets provide complementary information about UPI
transactions, customers, merchants, and chargebacks.

The initial assessment shows that the data contains duplicate records,
missing values, inconsistent formats, and non-standardized identifiers.
These issues can affect transaction metrics, fraud detection, merchant
risk analysis, and dataset joins.

Therefore, the next stage is to perform structured data rescue and
cleaning while preserving records that may themselves provide useful
fraud or data-quality signals.

**Next Step:** Data Cleaning and Standardization.